# Statistiques et prédictions en temps réel sur les locations de vélos

Ce TP traite un flux avec Structured Streaming : des relevés horaires de locations de vélos, qui arrivent jour après jour sous forme de fichiers. Vous y lisez le flux, écrivez une première requête, agrégez avec un état, calculez des totaux par fenêtre de temps, écartez les relevés en retard avec un watermark, suivez vos requêtes, écrivez des fichiers avec un point de reprise, puis entraînez un modèle par lots, l'appliquez au flux, enrichi d'une table et dédoublonné, et surveillez son erreur tranche par tranche.

Chaque exercice s'appuie sur le précédent : faites-les dans l'ordre. Spark tourne ici en mode local, sur les cœurs de la machine de Colab. Un vrai flux arriverait de Kafka ou d'un dossier alimenté par une autre application ; ici, la fonction `livrer` dépose elle-même les fichiers, ce qui vous laisse décider du moment où chaque journée arrive. La méthode `processAllAvailable` d'une requête attend que tout ce qui est arrivé soit traité : elle rend chaque cellule reproductible, là où une requête en production tourne sans fin.

## Installation et données

Les données sont celles de [Capital Bikeshare](https://archive.ics.uci.edu/dataset/275/bike+sharing+dataset), le service de vélos en libre-service de Washington, en 2011 et 2012 (Fanaee-T, UCI Machine Learning Repository, licence CC BY 4.0) : 17 379 lignes, une par heure. Les colonnes utiles ici : le jour (`dteday`) et l'heure (`hr`), le jour de la semaine (`weekday`, 0 pour dimanche), le jour ouvré (`workingday`), la météo (`weathersit`, de 1 pour un temps dégagé à 4 pour une forte pluie), la température (`temp`, divisée par 41 °C), l'humidité (`hum`), le vent (`windspeed`) et le nombre de locations de l'heure (`cnt`).

In [ ]:
!pip install pyspark
!wget -q -nc https://github.com/shuuchuu/datasets/raw/refs/heads/main/bike-sharing/hour.csv

In [ ]:
import os
import shutil

import pandas as pd
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

La session tourne en local sur deux cœurs. Deux réglages comptent pour un flux : `spark.sql.shuffle.partitions` fixe le nombre de partitions de l'état des agrégations, que le point de reprise fige au premier lancement (les 200 par défaut ralentiraient un si petit flux), et le fuseau horaire UTC aligne les fenêtres d'une journée sur minuit.

In [ ]:
spark = (
  SparkSession.builder.appName("velos-flux")
  .master("local[2]")
  .config("spark.sql.shuffle.partitions", "4")
  .config("spark.sql.session.timeZone", "UTC")
  .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

`velos` est la table entière, lue par lots : elle sert à entraîner le modèle et donne son schéma au flux. La fonction `livrer(jour)` simule l'arrivée des relevés d'un jour : elle les écrit dans le dossier `entrantes/`, d'un seul coup (le fichier est écrit sous un nom caché, puis renommé), pour que Spark ne lise jamais un fichier à moitié écrit. Son paramètre `nom` donne au fichier un autre nom que la date. La cellule vide aussi les dossiers des exercices, pour que le carnet puisse se relancer depuis le début.

In [ ]:
velos = spark.read.csv("hour.csv", header=True, inferSchema=True)
releves_pd = pd.read_csv("hour.csv")

for dossier in ["entrantes", "sortie", "reprise", "modele-velos"]:
  shutil.rmtree(dossier, ignore_errors=True)
os.makedirs("entrantes")


def livrer(jour, nom=None):
  lot = releves_pd[releves_pd["dteday"] == jour]
  cache = f"entrantes/.{jour}.csv"
  lot.to_csv(cache, index=False)
  os.replace(cache, f"entrantes/{nom or jour}.csv")
  return len(lot)


velos.printSchema()

## 1. Lire un flux de fichiers

Créez le flux `releves` avec `spark.readStream`, sur le dossier `entrantes` : des fichiers CSV avec une ligne d'en-tête, au schéma de `velos` (son attribut `schema`), un fichier par micro-lot (l'option `maxFilesPerTrigger`). Ajoutez-lui la colonne `heure`, l'horodatage du relevé : `dteday` converti en `timestamp`, plus `hr` heures. Affichez `releves.isStreaming`.

Le dossier est encore vide : pourquoi la création du flux réussit-elle quand même ? Et pourquoi `releves.count()` échouerait-il ?


**Documentation utile**

- [Sources de fichiers et leurs options](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#input-sources)
- [pyspark.sql.streaming.DataStreamReader.option](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamReader.option.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`maxFilesPerTrigger` prend un nombre : `option("maxFilesPerTrigger", 1)`. Un flux de fichiers ne devine pas son schéma : sans `schema(...)`, `readStream.csv` lève une erreur. La colonne `heure` s'écrit comme dans les diapositives, avec `F.make_dt_interval(hours=...)`.

```python
releves = (
  spark.readStream.schema(...)
  .option("header", True)
  .option("maxFilesPerTrigger", ...)
  .csv(...)
  .withColumn(
    "heure", F.col("dteday").cast("timestamp") + F.make_dt_interval(hours=...)
  )
)
releves.isStreaming
```

### Solution

In [ ]:
releves = (
  spark.readStream.schema(velos.schema)
  .option("header", True)
  .option("maxFilesPerTrigger", 1)
  .csv("entrantes")
  .withColumn(
    "heure", F.col("dteday").cast("timestamp") + F.make_dt_interval(hours=F.col("hr"))
  )
)
releves.isStreaming
# Rien n'est lu à la création : comme une transformation, readStream ne fait que
# décrire la source. Le schéma est donné, Spark n'a pas besoin de lire un fichier.
# releves.count() lèverait une AnalysisException : un flux n'a pas de fin, donc pas
# de nombre de lignes à rendre ; il ne s'exécute qu'à travers un puits
# (writeStream...start()).

## 2. Une première requête

Gardez du flux les heures de plus de 200 locations, avec leurs colonnes `heure`, `cnt` et `weathersit`, et écrivez-les dans le puits `memory`, sous le nom de requête `pointes`, en mode `append`. Démarrez la requête (`start`), puis livrez les 1er et 2 janvier 2012 (`livrer("2012-01-01")`…), attendez leur traitement (`processAllAvailable`) et affichez la table `pointes` (`spark.table`), triée par heure.

Livrez ensuite le 3 janvier et attendez son traitement : combien de lignes la table a-t-elle maintenant, et quel est le numéro du dernier micro-lot (`lastProgress["batchId"]`) ? Arrêtez enfin la requête (`stop`).


**Documentation utile**

- [pyspark.sql.streaming.DataStreamWriter.queryName](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.queryName.html)
- [pyspark.sql.streaming.StreamingQuery.processAllAvailable](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.processAllAvailable.html)
- [pyspark.sql.streaming.StreamingQuery.lastProgress](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.lastProgress.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le puits `memory` garde les lignes écrites dans une table du pilote, nommée d'après `queryName` : `spark.table("pointes")` la lit comme une table ordinaire. `lastProgress` est un dictionnaire ; ses micro-lots sont numérotés à partir de 0, un par fichier ici. Vous devez trouver 6 lignes après les deux premiers jours, 9 après le troisième.

```python
pointes = (
  releves.filter(...)
  .select(...)
  .writeStream.format("memory")
  .queryName(...)
  .outputMode(...)
  .start()
)
livrer(...)
livrer(...)
pointes.processAllAvailable()
spark.table(...).orderBy("heure").show(truncate=False)

livrer(...)
pointes.processAllAvailable()
print(spark.table(...).count(), pointes.lastProgress[...])
pointes.stop()
```

### Solution

In [ ]:
pointes = (
  releves.filter(F.col("cnt") > 200)
  .select("heure", "cnt", "weathersit")
  .writeStream.format("memory")
  .queryName("pointes")
  .outputMode("append")
  .start()
)
livrer("2012-01-01")
livrer("2012-01-02")
pointes.processAllAvailable()
spark.table("pointes").orderBy("heure").show(truncate=False)

livrer("2012-01-03")
pointes.processAllAvailable()
print(spark.table("pointes").count(), pointes.lastProgress["batchId"])
pointes.stop()
# 9 lignes, micro-lot 2 : trois fichiers, trois micro-lots (0, 1, 2). Le troisième
# n'a lu que le fichier du 3 janvier et n'a ajouté que ses 3 lignes : en mode
# append, une ligne écrite ne change plus.

## 3. Agréger avec un état

Calculez, pour les jours non ouvrés et les jours ouvrés (`workingday`), le nombre d'heures et la moyenne des locations par heure, arrondie à une décimale, dans le puits `memory` sous le nom `par_ouvre`, en mode `update`. Démarrez la requête, attendez son traitement et affichez la table, triée par `workingday` puis par nombre d'heures : le flux lit d'abord les trois fichiers déjà là, un par micro-lot, une nouvelle requête ne sachant rien de la précédente. Pourquoi les jours non ouvrés (0) y ont-ils deux lignes ?

Livrez le 4 janvier, attendez son traitement et affichez à nouveau la table : quelles lignes le micro-lot du 4 janvier ajoute-t-il, et combien en a-t-il écrit (`lastProgress["sink"]["numOutputRows"]`) ? Pourquoi le mode `append` serait-il refusé pour cette requête ? Arrêtez-la.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`groupBy(...).agg(F.count("*").alias("heures"), F.round(F.avg(...), 1).alias("moyenne"))`, comme sur une table. En mode `update`, le puits `memory` ajoute à sa table les lignes que chaque micro-lot écrit, sans remplacer les anciennes. Le 1er janvier est un dimanche, le 2 un lundi férié : après trois jours, la table a trois lignes, dont les jours non ouvrés sur 47 heures, à 90,3 locations par heure.

```python
par_ouvre = (
  releves.groupBy(...)
  .agg(...)
  .writeStream.format("memory")
  .queryName(...)
  .outputMode(...)
  .start()
)
par_ouvre.processAllAvailable()
spark.table(...).orderBy(..., ...).show()

livrer(...)
par_ouvre.processAllAvailable()
spark.table(...).orderBy(..., ...).show()
print(par_ouvre.lastProgress[...][...])
par_ouvre.stop()
```

### Solution

In [ ]:
par_ouvre = (
  releves.groupBy("workingday")
  .agg(F.count("*").alias("heures"), F.round(F.avg("cnt"), 1).alias("moyenne"))
  .writeStream.format("memory")
  .queryName("par_ouvre")
  .outputMode("update")
  .start()
)
par_ouvre.processAllAvailable()
spark.table("par_ouvre").orderBy("workingday", "heures").show()

livrer("2012-01-04")
par_ouvre.processAllAvailable()
spark.table("par_ouvre").orderBy("workingday", "heures").show()
print(par_ouvre.lastProgress["sink"]["numOutputRows"])
par_ouvre.stop()
# En mode update, un micro-lot n'écrit que les lignes qu'il a modifiées : le 1er
# janvier écrit les jours non ouvrés sur 24 heures, le 2 les réécrit sur 47 heures,
# le 3, jour ouvré, crée l'autre ligne. Le puits memory ajoute chaque version à sa
# table ; un puits qui met à jour par clé (une base, via foreachBatch) n'en garderait
# que la dernière. Le micro-lot du 4 janvier n'écrit qu'une ligne, les jours ouvrés
# sur 48 heures : la moyenne des jours non ouvrés n'a pas bougé. Spark n'a lu que les
# 24 lignes du nouveau fichier et les a ajoutées à l'état (un compte et une somme par
# clé), gardé entre les micro-lots. En mode append, une ligne écrite ne doit plus
# changer : une moyenne qui peut encore bouger n'est jamais définitive, Spark refuse
# donc la requête (AnalysisException, STREAMING_OUTPUT_MODE).

## 4. Des fenêtres de six heures

Calculez, par fenêtre fixe de 6 heures sur la colonne `heure` (`F.window`), le total des locations et la température moyenne arrondie à deux décimales, dans le puits `memory` sous le nom `par_6h`, en mode `complete`. Affichez le début et la fin de chaque fenêtre (les champs `start` et `end` de la colonne `window`), triées. Quelle tranche de la journée concentre les locations le dimanche 1er janvier, et les 3 et 4 janvier, jours ouvrés ?

Avec des fenêtres glissantes de 24 heures toutes les 6 heures, dans combien de fenêtres chaque relevé compterait-il ? Arrêtez la requête.


**Documentation utile**

- [pyspark.sql.functions.window](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.window.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`F.window("heure", "6 hours")` est une colonne de groupement, `F.col("window.start")` en lit le début. Le 1er janvier de 12 h à 18 h compte 1 282 locations.

```python
par_6h = (
  releves.groupBy(F.window(...))
  .agg(F.sum("cnt").alias("locations"), F.round(F.avg(...), 2).alias("temp"))
  .writeStream.format("memory")
  .queryName(...)
  .outputMode(...)
  .start()
)
par_6h.processAllAvailable()
(
  spark.table(...)
  .orderBy("window")
  .select(
    F.col("window.start").alias("debut"),
    F.col("window.end").alias("fin"),
    "locations",
    "temp",
  )
  .show(truncate=False)
)
par_6h.stop()
```

### Solution

In [ ]:
par_6h = (
  releves.groupBy(F.window("heure", "6 hours"))
  .agg(F.sum("cnt").alias("locations"), F.round(F.avg("temp"), 2).alias("temp"))
  .writeStream.format("memory")
  .queryName("par_6h")
  .outputMode("complete")
  .start()
)
par_6h.processAllAvailable()
(
  spark.table("par_6h")
  .orderBy("window")
  .select(
    F.col("window.start").alias("debut"),
    F.col("window.end").alias("fin"),
    "locations",
    "temp",
  )
  .show(truncate=False)
)
par_6h.stop()
# Le dimanche 1er janvier (et le lundi 2, férié), l'après-midi (12 h - 18 h) ; les
# 3 et 4 janvier, la matinée (6 h - 12 h, les trajets vers le travail). Avec une fenêtre
# de 24 heures qui glisse de 6 heures, chaque relevé compte dans 24 / 6 = 4 fenêtres.

## 5. Écarter les retards avec un watermark

Reprenez les fenêtres de 6 heures de l'exercice 4, avec cette fois le nombre d'heures et le total des locations, et un watermark de 12 heures sur `heure` (`withWatermark`), dans le puits `memory` sous le nom `par_tranche`, en mode `append`. Avant de démarrer la requête, prédisez les fenêtres qu'elle écrira : quatre journées sont arrivées, la plus grande `heure` vue est le 4 janvier à 23 h, et une fenêtre n'est écrite qu'une fois finie avant le watermark. Démarrez-la, attendez son traitement, affichez le début et la fin des fenêtres écrites, triées, et le watermark du dernier micro-lot (`lastProgress["eventTime"]`) : votre prédiction était-elle juste ?

Livrez le 5 janvier : quelles fenêtres s'ajoutent ? Livrez enfin un retardataire, les relevés du 2 janvier une seconde fois, sous un autre nom de fichier (`livrer("2012-01-02", "2012-01-02-retard")`) : la table gagne-t-elle des lignes ? Laissez la requête tourner, l'exercice suivant l'observe.

In [ ]:
# Votre code ici

### Conseils supplémentaires

`withWatermark` vient avant le `groupBy`, sur la colonne de la fenêtre : `releves.withWatermark("heure", "12 hours").groupBy(F.window("heure", "6 hours"))`. Le watermark vaut la plus grande `heure` vue moins le retard toléré. La fenêtre du 2 janvier de minuit à 6 h compte 5 heures (une heure manque dans les données) et 66 locations.

```python
par_tranche = (
  releves.withWatermark(...)
  .groupBy(...)
  .agg(F.count("*").alias("heures"), F.sum("cnt").alias("locations"))
  .writeStream.format("memory")
  .queryName(...)
  .outputMode(...)
  .start()
)
par_tranche.processAllAvailable()
(
  spark.table(...)
  .orderBy("window")
  .select(
    F.col("window.start").alias("debut"),
    F.col("window.end").alias("fin"),
    "heures",
    "locations",
  )
  .show(truncate=False)
)
print(par_tranche.lastProgress["eventTime"])

livrer(...)
par_tranche.processAllAvailable()
...

livrer("2012-01-02", "2012-01-02-retard")
par_tranche.processAllAvailable()
print(spark.table(...).count())
```

### Solution

In [ ]:
par_tranche = (
  releves.withWatermark("heure", "12 hours")
  .groupBy(F.window("heure", "6 hours"))
  .agg(F.count("*").alias("heures"), F.sum("cnt").alias("locations"))
  .writeStream.format("memory")
  .queryName("par_tranche")
  .outputMode("append")
  .start()
)
par_tranche.processAllAvailable()
(
  spark.table("par_tranche")
  .orderBy("window")
  .select(
    F.col("window.start").alias("debut"),
    F.col("window.end").alias("fin"),
    "heures",
    "locations",
  )
  .show(truncate=False)
)
print(par_tranche.lastProgress["eventTime"])

livrer("2012-01-05")
par_tranche.processAllAvailable()
(
  spark.table("par_tranche")
  .orderBy("window")
  .select(
    F.col("window.start").alias("debut"),
    F.col("window.end").alias("fin"),
    "heures",
    "locations",
  )
  .show(truncate=False)
)

livrer("2012-01-02", "2012-01-02-retard")
par_tranche.processAllAvailable()
print(spark.table("par_tranche").count())
# Après quatre jours, 13 fenêtres sont écrites, jusqu'à celle du 4 janvier de minuit
# à 6 h : le watermark vaut la plus grande heure vue (le 4 janvier à 23 h) moins
# 12 heures, le 4 janvier à 11 h ; la fenêtre de 6 h à midi finit après lui et peut
# encore recevoir des relevés. Le 5 janvier pousse le watermark au 5 à 11 h : les
# trois dernières fenêtres du 4 et la première du 5 sont écrites, puis retirées de
# l'état. Le retardataire est plus vieux que le watermark : Spark l'ignore, la table
# garde ses 17 fenêtres. Sans watermark, Spark aurait dû garder toutes les fenêtres,
# à jamais.

## 6. Suivre les requêtes

Affichez le nom des requêtes actives (`spark.streams.active` et leur attribut `name`) et l'état de `par_tranche` (`status`). Affichez ensuite, pour chacun de ses relevés de progression (`recentProgress`), le numéro du micro-lot (`batchId`) et le nombre de lignes lues (`numInputRows`) : quels relevés n'ont lu aucune ligne, et pourquoi ? Du dernier micro-lot qui a lu des lignes, affichez le numéro, le nombre de lignes lues et la durée totale (`durationMs["triggerExecution"]`, en millisecondes) : que vous apprend ce nombre de lignes sur le retardataire ? Pourquoi `lastProgress` ne suffirait-il pas ? Arrêtez enfin toutes les requêtes actives.


**Documentation utile**

- [pyspark.sql.streaming.StreamingQueryManager.active](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQueryManager.active.html)
- [pyspark.sql.streaming.StreamingQuery.status](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.status.html)
- [pyspark.sql.streaming.StreamingQuery.recentProgress](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.recentProgress.html)
- [Suivre une requête](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#monitoring-streaming-queries)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`spark.streams.active` est une liste de `StreamingQuery`. Seule `par_tranche` tourne encore : elle attend des données, et son `status` dit ce qu'elle fait à l'instant (`'Waiting for data to arrive'`, ou `'Getting offsets from ...'` quand elle interroge la source). `recentProgress` garde les derniers relevés de progression, du plus ancien au plus récent. Deux sortes de relevés n'ont lu aucune ligne : le micro-lot sans données que Spark lance quand le watermark avance, pour écrire les fenêtres finies, et, tant que la requête attend, un relevé vide toutes les 10 secondes, au numéro du prochain micro-lot. Passé 10 secondes d'attente, `lastProgress` est l'un d'eux : gardez les relevés dont `numInputRows` est positif.

```python
print([requete.name for requete in ...])
print(par_tranche.status)
print([(progres[...], progres[...]) for progres in par_tranche.recentProgress])
dernier = [progres for progres in par_tranche.recentProgress if ...][-1]
print(dernier[...], dernier[...], dernier[...][...])
for requete in ...:
  requete.stop()
```

### Solution

In [ ]:
print([requete.name for requete in spark.streams.active])
print(par_tranche.status)
print(
  [
    (progres["batchId"], progres["numInputRows"])
    for progres in par_tranche.recentProgress
  ]
)
dernier = [
  progres for progres in par_tranche.recentProgress if progres["numInputRows"] > 0
][-1]
print(
  dernier["batchId"], dernier["numInputRows"], dernier["durationMs"]["triggerExecution"]
)
for requete in spark.streams.active:
  requete.stop()
# Les micro-lots 4 et 6 n'ont rien lu : Spark les a lancés quand le watermark a
# avancé (après le 4, puis le 5 janvier), pour écrire les fenêtres finies et les
# retirer de l'état. Un relevé vide au numéro 8, répété toutes les 10 secondes, est
# celui d'une requête qui attend des données : dès le premier, lastProgress n'est plus
# le micro-lot 7. Celui-ci a bien lu les 23 lignes du retardataire : le watermark ne
# les empêche pas d'entrer, il les écarte de l'agrégation. Un nombre de lignes lues
# qui ne se retrouve pas dans les résultats signale des retards.

## 7. Écrire des fichiers avec un point de reprise

Écrivez la fonction `archiver()`, qui démarre une requête écrivant les colonnes `dteday`, `heure`, `cnt` et `temp` du flux en Parquet dans le dossier `sortie`, partitionné par jour (`partitionBy("dteday")`), avec le point de reprise `reprise` (l'option `checkpointLocation`), qui traite ce qui est arrivé puis s'arrête (`trigger(availableNow=True)`), attend sa fin (`awaitTermination`) et renvoie la requête.

Appelez-la, puis affichez les dossiers de `sortie` (`os.listdir`), le nombre de lignes lues par chaque micro-lot (`recentProgress`) et, de `sortie` lue par lots (`spark.read.parquet`), le nombre d'heures et le total des locations de chaque jour. Pourquoi le 2 janvier y compte-t-il double, alors qu'il gardait ses 1 951 locations à l'exercice 5 ? Livrez le 6 janvier et appelez-la à nouveau : combien de lignes ce second lancement lit-il, et pourquoi ?


**Documentation utile**

- [pyspark.sql.streaming.DataStreamWriter.partitionBy](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.partitionBy.html)
- [pyspark.sql.streaming.DataStreamWriter.trigger](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.trigger.html)
- [pyspark.sql.streaming.StreamingQuery.recentProgress](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.StreamingQuery.recentProgress.html)
- [Reprise après une panne](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#recovering-from-failures-with-checkpointing)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Le chemin du puits fichier se donne par `option("path", "sortie")`, et `partitionBy` s'écrit avant `start`, comme les options. `recentProgress` est la liste des derniers `lastProgress` : `[progres["numInputRows"] for progres in requete.recentProgress]`. Le premier lancement lit les six fichiers, retardataire compris, en six micro-lots : 142 lignes.

```python
def archiver():
  requete = (
    releves.select(...)
    .writeStream.format("parquet")
    .partitionBy(...)
    .option("path", ...)
    .option("checkpointLocation", ...)
    .trigger(...)
    .start()
  )
  requete.awaitTermination()
  return requete


requete = archiver()
print(sorted(os.listdir(...)))
print([progres[...] for progres in requete.recentProgress])
(
  spark.read.parquet(...)
  .groupBy(...)
  .agg(F.count("*").alias("heures"), F.sum("cnt").alias("locations"))
  .orderBy(...)
  .show()
)
livrer(...)
requete = archiver()
print(
  spark.read.parquet(...).count(),
  [progres[...] for progres in requete.recentProgress],
)
```

### Solution

In [ ]:
def archiver():
  requete = (
    releves.select("dteday", "heure", "cnt", "temp")
    .writeStream.format("parquet")
    .partitionBy("dteday")
    .option("path", "sortie")
    .option("checkpointLocation", "reprise")
    .trigger(availableNow=True)
    .start()
  )
  requete.awaitTermination()
  return requete


requete = archiver()
print(sorted(os.listdir("sortie")))
print([progres["numInputRows"] for progres in requete.recentProgress])
(
  spark.read.parquet("sortie")
  .groupBy("dteday")
  .agg(F.count("*").alias("heures"), F.sum("cnt").alias("locations"))
  .orderBy("dteday")
  .show()
)
livrer("2012-01-06")
requete = archiver()
print(
  spark.read.parquet("sortie").count(),
  [progres["numInputRows"] for progres in requete.recentProgress],
)
# Un dossier par jour (dteday=2012-01-01...), plus _spark_metadata, le journal des
# fichiers que chaque lot a écrits : spark.read.parquet ne lit que ceux-là. Le
# 2 janvier compte 46 heures et 3 902 locations : le puits fichier écrit chaque
# ligne à son arrivée, sans état ni watermark, et le retardataire est une seconde
# livraison des mêmes relevés. Le watermark de l'exercice 5 ne valait que pour son
# agrégation ; un dédoublonnage l'écarterait (exercice 9). Le second lancement ne lit
# que les 24 lignes du 6 janvier : le point de reprise garde les fichiers déjà
# traités et les lots déjà écrits. Une requête relancée après un arrêt, voulu ou non,
# reprend là où elle en était, sans doublon dans sortie. C'est le mode d'un job lancé
# chaque heure par un ordonnanceur : du streaming, mais sans cluster qui tourne en
# permanence.

## 8. Entraîner le modèle par lots

Le modèle prédit les locations d'une heure (`cnt`) d'après les colonnes de `VARIABLES`. Construisez un `Pipeline` qui assemble ces colonnes en un vecteur `features` (`VectorAssembler`), puis entraîne un `GBTRegressor` (`labelCol="cnt"`, 20 itérations, profondeur 6, `seed=0`). Entraînez-le sur les relevés de janvier à octobre 2011 (`yr` vaut 0 en 2011, `mnth` donne le mois), et mesurez son erreur absolue moyenne (MAE) sur ceux de novembre et décembre 2011 (`RegressionEvaluator`).

Entraînez-le enfin sur toute l'année 2011 et enregistrez le `PipelineModel` dans le dossier `modele-velos`.


**Documentation utile**

- [pyspark.ml.regression.GBTRegressor](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.regression.GBTRegressor.html)
- [pyspark.ml.evaluation.RegressionEvaluator](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.evaluation.RegressionEvaluator.html)

In [ ]:
VARIABLES = ["hr", "weekday", "workingday", "weathersit", "temp", "hum", "windspeed"]

# Votre code ici

### Conseils supplémentaires

Les itérations et la profondeur sont les paramètres `maxIter` et `maxDepth` ; l'évaluateur prend `labelCol="cnt", metricName="mae"`. L'erreur sur novembre et décembre 2011 est d'environ 47 locations par heure. Chaque entraînement prend de 30 secondes à une minute sur Colab.

```python
annee_2011 = velos.filter(F.col("yr") == 0)
entrainement = annee_2011.filter(...)
validation = annee_2011.filter(...)

pipeline = Pipeline(
  stages=[
    VectorAssembler(inputCols=..., outputCol="features"),
    GBTRegressor(
      featuresCol="features", labelCol=..., maxIter=..., maxDepth=..., seed=0
    ),
  ]
)
evaluateur = RegressionEvaluator(labelCol=..., metricName=...)
print(evaluateur.evaluate(pipeline.fit(...).transform(...)))

pipeline.fit(...).write().overwrite().save(...)
```

### Solution

In [ ]:
VARIABLES = ["hr", "weekday", "workingday", "weathersit", "temp", "hum", "windspeed"]
annee_2011 = velos.filter(F.col("yr") == 0)
entrainement = annee_2011.filter(F.col("mnth") <= 10)
validation = annee_2011.filter(F.col("mnth") > 10)

pipeline = Pipeline(
  stages=[
    VectorAssembler(inputCols=VARIABLES, outputCol="features"),
    GBTRegressor(
      featuresCol="features", labelCol="cnt", maxIter=20, maxDepth=6, seed=0
    ),
  ]
)
evaluateur = RegressionEvaluator(labelCol="cnt", metricName="mae")
print(evaluateur.evaluate(pipeline.fit(entrainement).transform(validation)))

pipeline.fit(annee_2011).write().overwrite().save("modele-velos")
# L'entraînement reste un job par lots : un fit sur un flux lèverait une
# AnalysisException, l'algorithme repasse plusieurs fois sur toutes les données.

## 9. Prédire sur le flux

Chargez le modèle (`PipelineModel.load`) et appliquez-le au flux `releves` (`transform`) : c'est le flux `predictions`. Créez aussi la table statique `meteo` des diapositives, qui nomme les météos 1 à 3 (« dégagé », « brume », « pluie ») : la météo 4, la forte pluie, n'y a pas de ligne.

Écartez d'abord de `predictions` les relevés reçus deux fois : un watermark de 4 jours sur `heure`, puis `dropDuplicatesWithinWatermark` sur `heure`. Joignez le résultat à `meteo` sur `weathersit`, et écrivez ses colonnes `heure`, `meteo`, `cnt` et `prediction` (arrondie à l'unité) dans le puits `memory` sous le nom `predictions`, en mode `append`. Attendez le traitement, affichez les premières heures, triées, et le nombre de lignes de la table. Arrêtez la requête.

Le flux relit les sept fichiers déjà livrés, retardataire compris, soit 166 lignes : combien la table en garde-t-elle ? Pourquoi un retard toléré de 4 jours, alors que le retardataire arrive juste après le 5 janvier ? Avec un jour, la table aurait autant de lignes : qu'est-ce qui écarterait alors le doublon ? Que deviennent les relevés de forte pluie dans la jointure ?


**Documentation utile**

- [pyspark.sql.DataFrame.dropDuplicatesWithinWatermark](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrame.dropDuplicatesWithinWatermark.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

`modele.transform(releves)` renvoie un flux, comme `releves.filter(...)`, et un flux se joint à une table statique comme deux tables : `join(meteo, "weathersit")`. Le dédoublonnage garde dans son état les `heure` déjà vues, le temps du retard toléré ; une ligne plus vieille que le watermark est écartée avant lui. Le retardataire arrive quand la plus grande `heure` vue est le 5 janvier à 23 h. Dans `recentProgress`, `stateOperators` dit qui a écarté ses lignes : `numRowsDroppedByWatermark` ou `customMetrics["numDroppedDuplicateRows"]`.

```python
modele = PipelineModel.load(...)
predictions = modele.transform(...)
meteo = spark.createDataFrame(
  [(1, "dégagé"), (2, "brume"), (3, "pluie")], ["weathersit", "meteo"]
)
requete = (
  predictions.withWatermark(...)
  .dropDuplicatesWithinWatermark(...)
  .join(...)
  .select("heure", "meteo", "cnt", F.round("prediction").alias("prediction"))
  .writeStream.format("memory")
  .queryName(...)
  .outputMode(...)
  .start()
)
requete.processAllAvailable()
spark.table(...).orderBy("heure").show(6)
print(spark.table(...).count())
requete.stop()
```

### Solution

In [ ]:
modele = PipelineModel.load("modele-velos")
predictions = modele.transform(releves)
meteo = spark.createDataFrame(
  [(1, "dégagé"), (2, "brume"), (3, "pluie")], ["weathersit", "meteo"]
)
requete = (
  predictions.withWatermark("heure", "4 days")
  .dropDuplicatesWithinWatermark(["heure"])
  .join(meteo, "weathersit")
  .select("heure", "meteo", "cnt", F.round("prediction").alias("prediction"))
  .writeStream.format("memory")
  .queryName("predictions")
  .outputMode("append")
  .start()
)
requete.processAllAvailable()
spark.table("predictions").orderBy("heure").show(6)
print(spark.table("predictions").count())
requete.stop()
# La table garde 143 lignes : les 23 du retardataire sont des doublons. Il arrive
# quand le flux a vu le 5 janvier à 23 h : pour que sa première heure, le 2 janvier
# à minuit, soit encore dans l'état du dédoublonnage, il faut un retard toléré de
# 4 jours : le watermark (le 1er janvier à 23 h) n'a pas atteint le 2 janvier.
# Avec un jour, le watermark (le 4 janvier à 23 h) l'écarterait avant le
# dédoublonnage, comme à l'exercice 5 (numRowsDroppedByWatermark : 23). Le
# dédoublonnage sert aux messages qu'une source rejoue dans le retard toléré (une
# reprise de Kafka). La jointure avec une table statique n'a pas d'état : chaque
# ligne du flux y cherche sa météo. La jointure interne perd sans erreur les relevés
# de forte pluie (météo 4) : aucun dans les jours livrés, trois sur les deux ans ;
# une jointure externe gauche (how="left") les garderait, sans libellé.

## 10. L'erreur tranche par tranche

Sur le flux `predictions`, avec un watermark de 12 heures sur `heure`, calculez par fenêtre de 6 heures le total des locations, le biais du modèle (la moyenne de `prediction - cnt`, arrondie à une décimale : positif quand il surestime) et son erreur absolue moyenne (la moyenne de `F.abs` du même écart, arrondie à une décimale). Écrivez-les dans le puits `memory` sous le nom `erreurs`, en mode `append`, attendez le traitement et affichez le début de chaque fenêtre et ses trois valeurs. Arrêtez la requête.

Le modèle sous-estime-t-il ou surestime-t-il les locations de janvier 2012, et dans quelles tranches de la journée se trompe-t-il le plus ? Que dit le biais que l'erreur absolue ne dit pas ? Pourquoi le modèle se trompe-t-il ainsi, et que faudrait-il faire ?

In [ ]:
# Votre code ici

### Conseils supplémentaires

Trois agrégats dans un même `agg` : `F.sum("cnt")`, `F.round(F.avg(F.col("prediction") - F.col("cnt")), 1)` et `F.round(F.avg(F.abs(...)), 1)`. Le 3 janvier de 6 h à midi, le modèle a un biais de −59,9 locations par heure. La table compte 21 fenêtres, et `show` n'en affiche que 20 par défaut.

```python
requete = (
  predictions.withWatermark(...)
  .groupBy(...)
  .agg(
    F.sum("cnt").alias("locations"),
    F.round(..., 1).alias("biais"),
    F.round(..., 1).alias("mae"),
  )
  .writeStream.format("memory")
  .queryName(...)
  .outputMode(...)
  .start()
)
requete.processAllAvailable()
(
  spark.table(...)
  .orderBy("window")
  .select(F.col("window.start").alias("debut"), "locations", "biais", "mae")
  .show(24, truncate=False)
)
requete.stop()
```

### Solution

In [ ]:
requete = (
  predictions.withWatermark("heure", "12 hours")
  .groupBy(F.window("heure", "6 hours"))
  .agg(
    F.sum("cnt").alias("locations"),
    F.round(F.avg(F.col("prediction") - F.col("cnt")), 1).alias("biais"),
    F.round(F.avg(F.abs(F.col("prediction") - F.col("cnt"))), 1).alias("mae"),
  )
  .writeStream.format("memory")
  .queryName("erreurs")
  .outputMode("append")
  .start()
)
requete.processAllAvailable()
(
  spark.table("erreurs")
  .orderBy("window")
  .select(F.col("window.start").alias("debut"), "locations", "biais", "mae")
  .show(24, truncate=False)
)
requete.stop()
# Le modèle sous-estime : en journée (de 6 h à minuit), son biais va de -24 à -60
# locations par heure les 3, 4 et 5 janvier, jours ouvrés, et vaut -36 le 2, férié ;
# les nuits restent justes. Seul le dimanche 1er janvier, de 6 h à 18 h, il surestime
# un peu. L'erreur absolue dit de combien il se trompe, le biais dans quel sens :
# quand les deux ont la même valeur, toutes ses erreurs vont dans le même sens. Le
# service a grandi entre 2011 et 2012, et le modèle, entraîné sur 2011 seulement,
# l'ignore. C'est une dérive : il faut le réentraîner, par lots, avec les relevés
# récents. Le flux sert à appliquer le modèle et à le surveiller, pas à l'entraîner.

## 11. Une alerte par micro-lot

Écrivez la fonction `surveiller(lot, id_lot)`, que `foreachBatch` appelle à chaque micro-lot avec ce micro-lot, un `DataFrame` statique, et son numéro. Elle calcule le nombre de lignes du lot et son erreur absolue moyenne, ajoute `(id_lot, nombre de lignes, erreur arrondie à une décimale)` à la liste `suivi` quand le lot n'est pas vide, et affiche une alerte quand l'erreur dépasse `SEUIL`. Démarrez une requête sur `predictions` avec `writeStream.foreachBatch(surveiller)`, attendez son traitement, arrêtez-la et affichez `suivi`.

Pourquoi `foreachBatch` plutôt qu'un puits ordinaire, pour une alerte ? `SEUIL` vaut 35 locations par heure, sous l'erreur de validation de l'exercice 8 (environ 47) : que se passerait-il sur la validation, puis l'été, et comment fixeriez-vous le seuil ?


**Documentation utile**

- [pyspark.sql.streaming.DataStreamWriter.foreachBatch](https://spark.apache.org/docs/latest/api/python/reference/pyspark.ss/api/pyspark.sql.streaming.DataStreamWriter.foreachBatch.html)

In [ ]:
SEUIL = 35
suivi = []

# Votre code ici

### Conseils supplémentaires

Dans `surveiller`, `lot` est un `DataFrame` ordinaire : `lot.agg(...).first()` renvoie une `Row`, dont les champs se lisent par leur nom (`resultat["mae"]`). Le premier lot (le 1er janvier) a une erreur de 29,4 ; les lots 2, 3, 4 et 6 dépassent le seuil.

```python
def surveiller(lot, id_lot):
  resultat = lot.agg(
    F.count("*").alias("lignes"),
    F.avg(F.abs(F.col("cnt") - F.col("prediction"))).alias("mae"),
  ).first()
  if resultat["lignes"] > 0:
    suivi.append(...)
    if ...:
      print(
        f"Lot {id_lot} : erreur moyenne de {resultat['mae']:.0f} vélos, à réentraîner ?"
      )


requete = predictions.writeStream.foreachBatch(...).start()
requete.processAllAvailable()
requete.stop()
suivi
```

### Solution

In [ ]:
SEUIL = 35
suivi = []


def surveiller(lot, id_lot):
  resultat = lot.agg(
    F.count("*").alias("lignes"),
    F.avg(F.abs(F.col("cnt") - F.col("prediction"))).alias("mae"),
  ).first()
  if resultat["lignes"] > 0:
    suivi.append((id_lot, resultat["lignes"], round(resultat["mae"], 1)))
    if resultat["mae"] > SEUIL:
      print(
        f"Lot {id_lot} : erreur moyenne de {resultat['mae']:.0f} vélos, à réentraîner ?"
      )


requete = predictions.writeStream.foreachBatch(surveiller).start()
requete.processAllAvailable()
requete.stop()
suivi
# foreachBatch donne chaque micro-lot comme une table ordinaire : toute l'API par lots
# s'y emploie (une agrégation sans état, first, un appel à un service d'alerte, une
# écriture JDBC ou vers plusieurs puits). La fonction tourne sur le pilote ; avec un
# point de reprise, id_lot permet d'ignorer un lot déjà traité avant une panne.
# Un seuil absolu de 35 est sous l'erreur de validation (47 en novembre et décembre
# 2011) : l'alerte y aurait déjà sonné, et sonnerait tout l'été, quand les locations
# et les erreurs montent ensemble. Un seuil se fixe d'après l'erreur de validation
# de la même saison, ou relatif au volume : par exemple, une alerte quand le total
# prévu du lot tombe sous les trois quarts du total observé.

## Fin

Arrêtez la session, qui libère les exécuteurs.

In [ ]:
spark.stop()